In [2]:
tsla = pd.read_csv('data/TSLA_features.csv', parse_dates=['date'], index_col='date').sort_index()

In [3]:
FEATURES = [
    'daily_return', 'weekly_return', 'ma_cross', 'dist_from_ma21', 'daily_range',
    'rsi_14', 'macd_hist', 'bb_position', 'volatility_7', 'volatility_20',
    'volume_change', 'volume_ratio',
    'lag_return_1', 'lag_return_3', 'lag_return_5',
    'month', 'is_earnings_week',
    'vix', 'is_major_event'
]
TARGET_CLS = 'target_direction'

In [4]:
tsla_cls = tsla[FEATURES + [TARGET_CLS]].dropna().copy()
print(tsla_cls.shape)

(2515, 20)


In [5]:
def walk_forward_splits(df, train_window, test_window=42, embargo=5):
    splits = []
    n = len(df)
    start = 0
    while start + train_window + embargo + test_window <= n:
        train_idx = list(range(start, start + train_window))
        test_idx  = list(range(start + train_window + embargo,
                               start + train_window + embargo + test_window))
        splits.append((train_idx, test_idx))
        start += test_window
    return splits

tsla_folds = walk_forward_splits(tsla_cls, train_window=59)
print("TSLA folds:", len(tsla_folds))

TSLA folds: 58


In [6]:
fold_scores = []
tsla_actual, tsla_pred = [], []
all_coefs = []

for train_idx, test_idx in tsla_folds:
    X_train = tsla_cls.iloc[train_idx][FEATURES]
    y_train = tsla_cls.iloc[train_idx][TARGET_CLS]
    X_test  = tsla_cls.iloc[test_idx][FEATURES]
    y_test  = tsla_cls.iloc[test_idx][TARGET_CLS]

    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)
    X_test_sc  = scaler.transform(X_test)

    lr = LogisticRegression(
        C=0.1, solver='lbfgs',
        class_weight='balanced', max_iter=1000, random_state=42
    )
    lr.fit(X_train_sc, y_train)
    y_p = lr.predict(X_test_sc)

    all_coefs.append(lr.coef_[0])

    cm = confusion_matrix(y_test, y_p, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0

    fold_scores.append({
        'accuracy':    accuracy_score(y_test, y_p),
        'f1':          f1_score(y_test, y_p, zero_division=0),
        'precision':   precision_score(y_test, y_p, zero_division=0),
        'recall':      recall_score(y_test, y_p, zero_division=0),
        'specificity': spec
    })
    tsla_actual.extend(y_test)
    tsla_pred.extend(y_p)

results_df = pd.DataFrame(fold_scores)
print(classification_report(tsla_actual, tsla_pred, zero_division=0))
print(results_df.describe())

              precision    recall  f1-score   support

           0       0.47      0.51      0.49      1170
           1       0.50      0.47      0.48      1266

    accuracy                           0.48      2436
   macro avg       0.49      0.49      0.48      2436
weighted avg       0.49      0.48      0.48      2436

        accuracy         f1  precision     recall  specificity
count  58.000000  58.000000  58.000000  58.000000    58.000000
mean    0.484811   0.430122   0.503278   0.482433     0.523905
std     0.077209   0.211193   0.183314   0.315810     0.312654
min     0.309524   0.000000   0.000000   0.000000     0.000000
25%     0.428571   0.300347   0.421559   0.212243     0.264035
50%     0.476190   0.475973   0.500000   0.473389     0.552727
75%     0.523810   0.603488   0.597727   0.756211     0.798319
max     0.666667   0.741935   1.000000   1.000000     1.000000


In [7]:
coef_df = pd.DataFrame(all_coefs, columns=FEATURES).mean()
coef_df = coef_df.sort_values()
print(coef_df.to_string())

rsi_14             -0.087966
daily_return       -0.076712
weekly_return      -0.074095
dist_from_ma21     -0.050663
ma_cross           -0.024893
is_earnings_week   -0.023761
volume_change      -0.012439
volatility_7       -0.011810
is_major_event     -0.008493
month               0.003624
daily_range         0.003945
lag_return_1        0.011272
bb_position         0.011837
macd_hist           0.015812
volume_ratio        0.016252
lag_return_3        0.020818
lag_return_5        0.023589
vix                 0.026488
volatility_20       0.026518


In [9]:
from itertools import product

C_values       = [0.001, 0.01, 0.1, 1, 10, 100]
penalty_values = ['l1', 'l2']

combos = list(product(C_values, penalty_values))
print(f'Grid: {len(C_values)} C values x {len(penalty_values)} penalties = {len(combos)} combinations')
print(f'Folds per combo: {len(tsla_folds)}')
print(f'Total LR fits: {len(combos) * len(tsla_folds)}')

grid_results = []

for C_val, pen in combos:
    fold_f1s, fold_accs, fold_precs, fold_recs, fold_specs = [], [], [], [], []

    for train_idx, test_idx in tsla_folds:
        X_train = tsla_cls.iloc[train_idx][FEATURES]
        y_train = tsla_cls.iloc[train_idx][TARGET_CLS]
        X_test  = tsla_cls.iloc[test_idx][FEATURES]
        y_test  = tsla_cls.iloc[test_idx][TARGET_CLS]

        scaler = StandardScaler()
        X_train_sc = scaler.fit_transform(X_train)
        X_test_sc  = scaler.transform(X_test)

        lr = LogisticRegression(C=C_val, solver='liblinear', l1_ratio=1.0 if pen == 'l1' else 0.0,
                                 class_weight='balanced', max_iter=1000, random_state=42
                                                                                        )
        
        lr.fit(X_train_sc, y_train)
        y_p = lr.predict(X_test_sc)

        fold_f1s.append(f1_score(y_test, y_p, zero_division=0))
        fold_accs.append(accuracy_score(y_test, y_p))
        fold_precs.append(precision_score(y_test, y_p, zero_division=0))
        fold_recs.append(recall_score(y_test, y_p, zero_division=0))

        cm = confusion_matrix(y_test, y_p, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        fold_specs.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

    grid_results.append({
        'C':           C_val,
        'penalty':     pen,
        'f1':          np.mean(fold_f1s),
        'accuracy':    np.mean(fold_accs),
        'precision':   np.mean(fold_precs),
        'recall':      np.mean(fold_recs),
        'specificity': np.mean(fold_specs),
    })
    print(f'  C={str(C_val):<7}  penalty={pen:<3}  '
          f'F1={np.mean(fold_f1s):.3f}  acc={np.mean(fold_accs):.3f}  '
          f'prec={np.mean(fold_precs):.3f}  rec={np.mean(fold_recs):.3f}  spec={np.mean(fold_specs):.3f}')

lr_grid_df = pd.DataFrame(grid_results)

Grid: 6 C values x 2 penalties = 12 combinations
Folds per combo: 58
Total LR fits: 696
  C=0.001    penalty=l1   F1=0.000  acc=0.480  prec=0.000  rec=0.000  spec=1.000
  C=0.001    penalty=l2   F1=0.428  acc=0.481  prec=0.506  rec=0.459  spec=0.525
  C=0.01     penalty=l1   F1=0.000  acc=0.480  prec=0.000  rec=0.000  spec=1.000
  C=0.01     penalty=l2   F1=0.427  acc=0.483  prec=0.495  rec=0.458  spec=0.533
  C=0.1      penalty=l1   F1=0.020  acc=0.479  prec=0.016  rec=0.026  spec=0.973
  C=0.1      penalty=l2   F1=0.429  acc=0.484  prec=0.501  rec=0.479  spec=0.525
  C=1        penalty=l1   F1=0.434  acc=0.489  prec=0.511  rec=0.488  spec=0.527
  C=1        penalty=l2   F1=0.426  acc=0.490  prec=0.514  rec=0.485  spec=0.540
  C=10       penalty=l1   F1=0.460  acc=0.500  prec=0.500  rec=0.535  spec=0.495
  C=10       penalty=l2   F1=0.448  acc=0.499  prec=0.504  rec=0.519  spec=0.515
  C=100      penalty=l1   F1=0.457  acc=0.498  prec=0.486  rec=0.542  spec=0.481
  C=100      penalty=

In [10]:
print('Top 5 by F1:')
print(lr_grid_df.sort_values('f1', ascending=False).head().to_string(index=False))

print('\nTop 5 by Accuracy:')
print(lr_grid_df.sort_values('accuracy', ascending=False).head().to_string(index=False))

print('\nTop 5 by Precision:')
print(lr_grid_df.sort_values('precision', ascending=False).head().to_string(index=False))

print('\nTop 5 by Specificity:')
print(lr_grid_df.sort_values('specificity', ascending=False).head().to_string(index=False))

print('\nBaseline (C=0.1, l2):')
print(lr_grid_df[(lr_grid_df['C'] == 0.1) & (lr_grid_df['penalty'] == 'l2')].to_string(index=False))

Top 5 by F1:
    C penalty       f1  accuracy  precision   recall  specificity
 10.0      l1 0.460181  0.499589   0.499621 0.534591     0.495227
100.0      l1 0.456592  0.498358   0.486243 0.541898     0.481403
100.0      l2 0.453079  0.497537   0.483088 0.535080     0.488503
 10.0      l2 0.448106  0.498768   0.504478 0.518855     0.514808
  1.0      l1 0.434108  0.488506   0.511177 0.488416     0.526618

Top 5 by Accuracy:
    C penalty       f1  accuracy  precision   recall  specificity
 10.0      l1 0.460181  0.499589   0.499621 0.534591     0.495227
 10.0      l2 0.448106  0.498768   0.504478 0.518855     0.514808
100.0      l1 0.456592  0.498358   0.486243 0.541898     0.481403
100.0      l2 0.453079  0.497537   0.483088 0.535080     0.488503
  1.0      l2 0.425627  0.490148   0.513569 0.484715     0.539937

Top 5 by Precision:
     C penalty       f1  accuracy  precision   recall  specificity
 1.000      l2 0.425627  0.490148   0.513569 0.484715     0.539937
 1.000      l1 0.434